In version 0, we interpolated the model state to the desired pressure levels at each individual time
In version 1, we will compute an average model state over the desired time period, and interpolate to the desired pressure levels only one time using that average state

In [1]:

import numpy as np
import glob
import os
import netCDF4 as nc4
import pickle as pkl
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import matplotlib.tri as tri
import math
from scipy.interpolate import interp1d
import sim_name_remap as map
import plotter_config as pc


In [2]:

def get_struc_var(FILES, var, prs_lvls, trp_lvls):

    file0 = nc4.Dataset(FILES[0])        
    hyam = file0.variables['hyam'][:] 
    hybm = file0.variables['hybm'][:]    
    lon = file0.variables['lon'][:]
    lat = file0.variables['lat'][:]
    
    var_strs = pc.var_info[var]['vars']

    # This is a flag that this is a 3d variable (no height info)
    if prs_lvls[0] == 0: flag3d = 1
    else: flag3d = 0
        
    var_array = [] 
    PS_array = []
    ####var_array_trplev = np.zeros((1,len(trp_lvls),len(lat),len(lon)))-999
    
    for f in range(len(FILES)):

        if f % 50 == 0: print('*** File ', f+1, ' of ', len(FILES))

        file = nc4.Dataset(FILES[f])
        if len(var_strs) > 1:       
            curr_var_array = []
            for sub_var in var_strs:
                curr_var_array.append(file.variables[sub_var][:])
            curr_var_array = np.asarray(curr_var_array)
            curr_var_array = np.nansum(curr_var_array, axis=0)
        # If not, read the variable directly
        else:
            curr_var_array = file.variables[var_strs[0]][:]

        if f == 0: 
            var_array.append(curr_var_array)
            var_array = var_array[0]
            PS_array.append(file.variables['PS'][:])
            PS_array = PS_array[0]
        else: 
            var_array = np.append(var_array, curr_var_array, axis=0)
            PS_array = np.append(PS_array, file.variables['PS'][:], axis=0)
                               
        #### TROP_P = file.variables['TROP_P'][:]  Everything with 4 #s will allow the trop rel capability to run again
        #GPH = file.variables['Z3'][:]

    #print(np.shape(var_array))
    #print(np.shape(PS_array))

    # Compute the mean, then we will interpolate to the pressure level we desire
    var_array_mean = np.nanmean(var_array, axis=0)
    PS_array_mean = np.nanmean(PS_array, axis=0)

    # 4d variable (t,lev,lt,ln)
    if flag3d == 0: 

        var_array_plev = np.zeros((len(prs_lvls),len(lat), len(lon)))-999
        
        for ln in range(len(lon)):
            for lt in range(len(lat)):

                curr_plev = (hyam*1.e5 + hybm*PS_array_mean[lt,ln])/100.
                curr_varprof = np.squeeze(var_array_mean[:,lt,ln])
                #### curr_TROP_P = TROP_P[t,lt,ln]/100.
                #curr_gphprof = np.squeeze(GPH[t,:,lt,ln])
            
                # Fill the pressure level array                
                pvar_f = interp1d(curr_plev, np.squeeze(curr_varprof), bounds_error=False, fill_value=np.nan)
                var_array_plev[:,lt,ln] = pvar_f(prs_lvls)

    # 3d variable (t,lt,ln)
    else: 

        var_array_plev = var_array_mean[np.newaxis,:,:]

    print(np.shape(var_array_plev))
        
    return var_array_plev, -999, lon, lat, -999, -999   #### var_array_trplev[1:,:,:,:]


In [3]:

def get_col_CO(FILES, var, prs_lvls, trp_lvls):

    file0 = nc4.Dataset(FILES[0])        
    hyam = file0.variables['hyam'][:] 
    hybm = file0.variables['hybm'][:]    
    lon = file0.variables['lon'][:]
    lat = file0.variables['lat'][:]
    
    var_strs = pc.var_info[var]['vars']

    # This is a flag that this is a 3d variable (no height info)
    if prs_lvls[0] == 0: flag3d = 1
    else: flag3d = 0
        
    var_array = [] 
    PDELDRY_array = []

    ####var_array_trplev = np.zeros((1,len(trp_lvls),len(lat),len(lon)))-999
    
    for f in range(len(FILES)):

        if f % 50 == 0: print('*** File ', f+1, ' of ', len(FILES))

        file = nc4.Dataset(FILES[f])
        if len(var_strs) > 1:       
            curr_var_array = []
            for sub_var in var_strs:
                curr_var_array.append(file.variables[sub_var][:])
            curr_var_array = np.asarray(curr_var_array)
            curr_var_array = np.nansum(curr_var_array, axis=0)
        # If not, read the variable directly
        else:
            curr_var_array = file.variables[var_strs[0]][:]
            curr_PDELDRY = file.variables['PDELDRY'][:]

        if f == 0: 
            var_array.append(curr_var_array)
            var_array = var_array[0]
            
        else: 
            var_array = np.append(var_array, curr_var_array, axis=0)
                               
        #### TROP_P = file.variables['TROP_P'][:]  Everything with 4 #s will allow the trop rel capability to run again
        #GPH = file.variables['Z3'][:]

    #print(np.shape(var_array))
    #print(np.shape(PS_array))

    # Compute the mean, then we will interpolate to the pressure level we desire
    var_array_mean = np.nanmean(var_array, axis=0)
    PS_array_mean = np.nanmean(PS_array, axis=0)

    # 4d variable (t,lev,lt,ln)
    if flag3d == 0: 

        var_array_plev = np.zeros((len(prs_lvls),len(lat), len(lon)))-999
        
        for ln in range(len(lon)):
            for lt in range(len(lat)):

                curr_plev = (hyam*1.e5 + hybm*PS_array_mean[lt,ln])/100.
                curr_varprof = np.squeeze(var_array_mean[:,lt,ln])
                #### curr_TROP_P = TROP_P[t,lt,ln]/100.
                #curr_gphprof = np.squeeze(GPH[t,:,lt,ln])
            
                # Fill the pressure level array                
                pvar_f = interp1d(curr_plev, np.squeeze(curr_varprof), bounds_error=False, fill_value=np.nan)
                var_array_plev[:,lt,ln] = pvar_f(prs_lvls)

    # 3d variable (t,lt,ln)
    else: 

        var_array_plev = var_array_mean[np.newaxis,:,:]

    print(np.shape(var_array_plev))
        
    return var_array_plev, -999, lon, lat, -999, -999   #### var_array_trplev[1:,:,:,:]


In [4]:
                            # Fill the tropopause-relative array 
                            
                            #trop_index_frac = plev_f(curr_TROP_P)
                            #### curr_deltaplev = curr_plev - curr_TROP_P
        
                            #### f_deltap = interp1d(curr_deltaplev, curr_varprof, bounds_error=False, fill_value=-999) 
                            
                            #### curr_var_array_trplev[t,:,lt,ln] = f_deltap(trp_lvls)
                        
                            #gph_f = interp1d(np.arange(len(curr_gphprof)), curr_gphprof, bounds_error=False, fill_value=-999)
                            #curr_TROP_gph = gph_f(trop_index_frac)
            
                            #curr_deltazlev = (curr_gphprof - curr_TROP_gph)/1.0e3
                            #f_deltaz = interp1d(curr_deltazlev, curr_varprof, bounds_error=False, fill_value=-999)  

In [5]:

#vars = ['CO','ISOP','NOx','OH','SO2','SO4','O3','ZMMU','T','CO_CHML','CO_CHMP']    # 'E90',
#vars = ['LNO_COL_PROD','SFISOP']
vars = ['LNO_PROD']
#vars = ['CO','E90'] #,'NOx','SO2','SO4',]
#vars = ['NOx']#'SO2','SO4']
#vars = ['CAPE']
#vars = ['LNO_COL_PROD']
#vars = ['SFISOP']
#vars = ['CO_CHML','CO_CHMP']
#vars = ['Target_T']
#sims = ['2022_MERRA2_ZM_h0','2022_ERA5_ZMKE_h0','2022_MERRA2_ZMKE_h0','2022_ERA5_ZM_h0']
##sims = ['2022_MERRA2_ZM','2022_ERA5_ZMKE','2022_MERRA2_ZMKE','2022_ERA5_ZM']#'2022_ERA5_ZM_3hr','2022_ERA5_ZMKE_3hr']
#sims = ['2022_ERA5_ZM_lno_h0','2022_ERA5_ZMKE_lno_h0','2022_MERRA2_ZMKE_lno_h0','2022_MERRA2_ZM_h0']
#sims = ['2022_M2-3hr_h0','2022_E5-3hr_h0','2022_E5-1hr_h0','2022_E5-1hr_ZMKE_h0','2022_E5-1hr_Qnudge_h0','2022_E5-1hr_ZMKE_h0']
##sims = ['2022_MERRA2_ZM','2022_ERA5_ZM']
#sims = ['2022_ERA5_ZMKE']
#sims = ['2022_M2-weak','2022_M2-strong','2022_E5-weak','2022_E5-strong']#,'2022_FreeRun']
sims = ['2022_M2-weak_h0','2022_M2-strong_h0','2022_E5-weak_h0','2022_E5-strong_h0','2022_FreeRun_h0']
#sims = ['ERA5_f09_32L','MERRA-2_f09_32L']

prs_lvls = [70.,100.,150.,200.,300.,400.,850.]
##prs_lvls = [100,200,250,300,400]  # For ZMMU only, probably
#prs_lvls = [0]

trp_lvls = [-1,0,1]     # These aren't actually used at the moment anyways

SAVEDIR = '/glade/u/home/wsmith/NudgeComparison_scripts/vars/'

#date0 = '2022-08-18'  # For 3hour files, that now go the whole year, but this is prohibitively expensive
#date1 = '2022-08-18'

date0 = '2022-08'   # For h0 files
date1 = '2022-08'


In [6]:

for sim in sims: 

    sim_path = map.remap[sim]
    FILES = sorted(glob.glob(sim_path))

    FILES0 = [item for item in FILES if date0 in item][0]
    find0 = list(FILES).index(FILES0)
    FILES1 = [item for item in FILES if date1 in item][0]
    find1 = list(FILES).index(FILES1)
    
    FILES = FILES[find0:find1+1]

    for v in range(len(vars)):
    
        var = vars[v]

        SAVEFILE = SAVEDIR + sim + '_' + var + '_AvgMap_plev_' + date0 + '_' + date1 + '.pkl'

        #####if not os.path.isfile(SAVEFILE): 

        print('Saving ' + var + ' from ' + sim)
        
        if var == 'XCO': var_array_plev, var_array_trplev, lon, lat, date, datesec = get_col_CO(FILES, var, prs_lvls, trp_lvls)
        else: var_array_plev, var_array_trplev, lon, lat, date, datesec = get_struc_var(FILES, var, prs_lvls, trp_lvls)
    
        with open(SAVEFILE, 'wb') as f:  
            pkl.dump((var_array_plev, prs_lvls, lon, lat, date, datesec), f)  
    
        # This part of the code is not functioning because of the above subroutines
        #with open(SAVEDIR + sim + '_' + var + '_MapData_trplev.pkl', 'wb') as f:  
        #    pkl.dump((var_array_trplev, trp_lvls, lon, lat, date, datesec), f)

        #####else: print(var + ' from ' + sim + ' already has been saved, skipping it...')
            

Saving LNO_PROD from 2022_M2-weak_h0
*** File  1  of  1
(7, 192, 288)
Saving LNO_PROD from 2022_M2-strong_h0
*** File  1  of  1
(7, 192, 288)
Saving LNO_PROD from 2022_E5-weak_h0
*** File  1  of  1
(7, 192, 288)
Saving LNO_PROD from 2022_E5-strong_h0
*** File  1  of  1
(7, 192, 288)
Saving LNO_PROD from 2022_FreeRun_h0
*** File  1  of  1
(7, 192, 288)
